### Часть 1: Материалы
1. Загрузить структуру материала по его названию ("MnFe2O4").  
2. Проанализировать состав и элементы материала.  
3. Вычислить параметры решётки и плотность.

In [13]:
import yaml
def load_api_key_from_config(path="config.yaml"):
    with open(path, "r") as file:
        config = yaml.safe_load(file)
        return config["materials_project"]["api_key"]

api_key = load_api_key_from_config()

In [14]:

from pymatgen.ext.matproj import MPRester
from mp_api.client import MPRester
from pymatgen.analysis.structure_analyzer import SpacegroupAnalyzer

mpr = MPRester(api_key)
docs = mpr.materials.summary.search(formula="MnFe2O4")

doc = docs[0]
structure = doc.structure
print(structure)

Retrieving SummaryDoc documents: 100%|██████████| 6/6 [00:00<?, ?it/s]

Full Formula (Mn10 Fe20 O40)
Reduced Formula: Mn(FeO2)2
abc   :   6.150494   6.100717  30.396005
angles:  60.279723  59.998079  59.837294
pbc   :       True       True       True
Sites (70)
  #  SP           a         b         c    magmom
---  ----  --------  --------  --------  --------
  0  Mn    0.125585  0.12847   0.02481      4.639
  1  Mn    0.002381  0.500143  0.099514     3.989
  2  Mn    0.128669  0.127284  0.224312     4.634
  3  Mn    0.128352  0.122436  0.425415     4.631
  4  Mn    0.000682  0.497553  0.50056      4.116
  5  Mn    0.12062   0.124349  0.627158     4.63
  6  Mn    0.002588  0.501439  0.299683     4.492
  7  Mn    0.122237  0.127152  0.825253     4.642
  8  Mn    0.998346  0.502359  0.700177     4.095
  9  Mn    0.878815  0.874438  0.9738       4.649
 10  Fe    0.502299  0.001466  0.099341     4.369
 11  Fe    0.500029  0.503541  0.999434     4.38
 12  Fe    0.500882  0.499626  0.099371     4.367
 13  Fe    0.497122  0.005267  0.299203     4.371
 14  Fe    0

In [15]:
composition = structure.composition
print("\nСостав:", composition.formula)
print("Элементы:", composition.elements)
print("Доли элементов:", composition.fractional_composition.as_dict())


Состав: Mn10 Fe20 O40
Элементы: [Element Mn, Element Fe, Element O]
Доли элементов: {'Mn': 0.14285714285714285, 'Fe': 0.2857142857142857, 'O': 0.5714285714285714}


In [16]:
lattice = structure.lattice
print("\nПараметры решётки:")
print(f"a = {lattice.a:.3f} , b = {lattice.b:.3f} , c = {lattice.c:.3f} ")
print(f"α = {lattice.alpha:.2f}, β = {lattice.beta:.2f}, γ = {lattice.gamma:.2f}")
print(f"\nПлотность = {structure.density:.3f} г/см3")

sga = SpacegroupAnalyzer(structure)
print("Пространственная группа:", sga.get_space_group_symbol())


Параметры решётки:
a = 6.150 , b = 6.101 , c = 30.396 
α = 60.28, β = 60.00, γ = 59.84

Плотность = 4.745 г/см3
Пространственная группа: P1


### Часть 2: Последовательности ДНК/РНК
1. Создать или выбрать последовательности по названиям генов или произвольные последовательности.  
2. Вычислить GC-состав каждой последовательности.  
3. Получить обратную комплементарную последовательность и транскрибировать в РНК. 


In [18]:
from dataclasses import dataclass
from typing import Dict, List
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
import pandas as pd

user_sequences: Dict[str, str] = {
    "seq_1": "ATGCGTATCGATCGATCGTTTGGGCCCATTA",
    "seq_2": "GGGTTTAAACCCGGGTTTAAACCC",
}

gene_db: Dict[str, str] = {
    "TP53":  "ATGGAGGAGCCGCAGTCAGATCCTAGCGTCGAGCCCCCTCTGAGTCAGGAA",
    "BRCA1": "ATGGATTTATCTGCTCTTCGGAAAAGTTGTCATTTTATGATGGAAGCTG",
    "EGFR":  "ATGCGACCCTCCGGGACGGCCGGGGCAGCGCTCCTGGACTATGTCCG",
}

gene_names: List[str] = ["TP53", "EGFR"]


In [19]:
def clean_to_dna(seq: str) -> Seq:
    
    s = "".join(c for c in seq.upper() if c in {"A", "C", "G", "T", "U"})
    s = s.replace("U", "T")
    return Seq(s)

@dataclass
class SeqItem:
    name: str
    dna: Seq

def collect_sequences(user_seqs: Dict[str, str], genes: List[str], gene_db: Dict[str, str]) -> List[SeqItem]:
    out: List[SeqItem] = []
    for name, s in user_seqs.items():
        dna = clean_to_dna(s)
        if len(dna) > 0:
            out.append(SeqItem(name=name, dna=dna))
    for g in genes:
        if g in gene_db:
            dna = clean_to_dna(gene_db[g])
            if len(dna) > 0:
                out.append(SeqItem(name=g, dna=dna))
    return out

def analyze(items: List[SeqItem]) -> pd.DataFrame:
    rows = []
    for it in items:
        dna = it.dna
        gc_pct = round(gc_fraction(dna) * 100.0, 2) if len(dna) else 0.0
        revc = dna.reverse_complement()
        rna  = dna.transcribe()  
        rows.append({
            "name": it.name,
            "length": len(dna),
            "GC_%": gc_pct,
            "DNA": str(dna),
            "reverse_complement": str(revc),
            "RNA_transcribed": str(rna),
        })
    return pd.DataFrame(rows, columns=["name", "length", "GC_%", "DNA", "reverse_complement", "RNA_transcribed"])

In [22]:
items = collect_sequences(user_sequences, gene_names, gene_db)
df = analyze(items)

print(df[["name", "length", "GC_%"]])
print("\n Первая запись")
if not df.empty:
    first = df.iloc[0]
    print("DNA:", first["DNA"])
    print("rev-compl:", first["reverse_complement"])
    print("RNA:", first["RNA_transcribed"])

    name  length   GC_%
0  seq_1      31  48.39
1  seq_2      24  50.00
2   TP53      51  60.78
3   EGFR      47  72.34

 Первая запись
DNA: ATGCGTATCGATCGATCGTTTGGGCCCATTA
rev-compl: TAATGGGCCCAAACGATCGATCGATACGCAT
RNA: AUGCGUAUCGAUCGAUCGUUUGGGCCCAUUA


### Часть 3: Белки
1. Выбрать белок альфа Амилаза по названию или UniProt ID и получить его последовательность аминокислот.  
2. Вычислить физико-химические дескрипторы белка (состав аминокислот, частоты дипептидов). 

In [23]:
import requests
from collections import Counter

name_or_id = "alpha amylase"   
NAME2ID = {
    "alpha amylase": "P04745",           
    "salivary alpha amylase": "P04745",
    "pancreatic alpha amylase": "P19961"
}

AA20 = "ACDEFGHIKLMNPQRSTVWY"

def fetch_uniprot_seq(uid: str) -> str:
    url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    lines = [ln.strip() for ln in r.text.splitlines() if not ln.startswith(">")]
    seq = "".join(lines).upper()
    return "".join(ch for ch in seq if ch in AA20)

def get_sequence(name_or_id: str) -> str:
    key = name_or_id.strip().lower()
    uid = NAME2ID.get(key, name_or_id.strip())
    try:
        seq = fetch_uniprot_seq(uid)
        if seq:
            print(f" Получена последовательность UniProt {uid}, длина: {len(seq)} aa")
            return seq
    except Exception as e:
        print(f" Не удалось скачать по '{uid}': {e}")
    return "MKSLVLSLLLLSLVSSAQGALAAPLLQASALSSSSTYSSTRSSSTYQVPQGHLGPHNQ"  

def aa_composition(seq: str):
    L = len(seq)
    counts = Counter(seq)
    counts = {aa: counts.get(aa, 0) for aa in AA20}
    freqs  = {aa: (counts[aa] / L if L else 0.0) for aa in AA20}
    return counts, freqs

def dipeptide_frequencies(seq: str):
    L = len(seq)
    if L < 2:
        return {}
    di_counts = Counter(seq[i:i+2] for i in range(L-1))
    denom = L - 1
    return {d: di_counts[d]/denom for d in (a+b for a in AA20 for b in AA20)}


seq = get_sequence(name_or_id)
aa_counts, aa_freqs = aa_composition(seq)
di_freqs = dipeptide_frequencies(seq)

In [25]:

print("\n Состав аминокислот ")
for aa in AA20:
    print(f"{aa}: {aa_counts[aa]:4d}  |  {aa_freqs[aa]:.4f}")

print("\n Примеры дипептидов с ненулевой частотой (первые 20)  ")
nonzero = [(d, f) for d, f in di_freqs.items() if f > 0]
for d, f in nonzero[:20]:
    print(f"{d}: {f:.6f}")



 Состав аминокислот 
A:    6  |  0.1034
C:    0  |  0.0000
D:    0  |  0.0000
E:    0  |  0.0000
F:    0  |  0.0000
G:    3  |  0.0517
H:    2  |  0.0345
I:    0  |  0.0000
K:    1  |  0.0172
L:   12  |  0.2069
M:    1  |  0.0172
N:    1  |  0.0172
P:    3  |  0.0517
Q:    5  |  0.0862
R:    1  |  0.0172
S:   15  |  0.2586
T:    3  |  0.0517
V:    3  |  0.0517
W:    0  |  0.0000
Y:    2  |  0.0345

 Примеры дипептидов с ненулевой частотой (первые 20)  
AA: 0.017544
AL: 0.035088
AP: 0.017544
AQ: 0.017544
AS: 0.017544
GA: 0.017544
GH: 0.017544
GP: 0.017544
HL: 0.017544
HN: 0.017544
KS: 0.017544
LA: 0.017544
LG: 0.017544
LL: 0.070175
LQ: 0.017544
LS: 0.052632
LV: 0.035088
MK: 0.017544
NQ: 0.017544
PH: 0.017544
